# RUNG 12 — pMHC structural discriminability: measure beta, certify relay targets (T4 GPU)

The RUNG-12P bridge unlocked RUNG-11's `tcr_dependent` handles **if** the TCR's mut-vs-WT cross-bind `beta` is low — but `beta` was *swept*. This run **measures** a per-handle `beta` from **ESM-2** (sequence) + **ESMFold** (structure, best-effort) + the RUNG-11 NetMHCpan binding differential + physicochemistry, then re-runs the bridge with the measured `beta` → a **ranked, certified target list**.

**Robustness:** the measured `beta` core (binding `M` + physicochemical `P` + ESM-2 `Z`) needs no folding. **ESMFold (the structural exposure `E`) is best-effort** — its peptide-into-groove docking is the soft part; if its install/fold hiccups, the run still produces a valid result (E falls back to a position prior). Use **T4 GPU**. Resumable Drive PDBs.

Output: `rung12_pmhc.json` + `rung12_pmhc.png`. Bundle with `archive_colab_run.py --commit`.

In [ ]:
#@title Cell 1 — clone / pull repo
import os
from pathlib import Path
REPO = Path('/content/cancer-recon-apoptosis')
if REPO.exists():
    !cd {REPO} && rm -f runs/logs/*.log && git fetch origin && git reset --hard origin/main
else:
    !git clone https://github.com/AnshumanAtrey/cancer-recon-apoptosis.git {REPO}
os.chdir(REPO)
!git log --oneline -1
print('[CELL 1] ✓')

In [ ]:
#@title Cell 2 — mount Drive (resumable PDBs) + PREP (fetch HLA grooves, write ESMFold inputs)
import os
try:
    from google.colab import drive; drive.mount('/content/drive')
    cd = '/content/drive/MyDrive/cancer-recon'; os.makedirs(cd, exist_ok=True)
    os.environ['RUNG12_WORK'] = f'{cd}/rung12_work'; os.makedirs(os.environ['RUNG12_WORK'], exist_ok=True)
    print('[CELL 2] Drive mounted; work dir', os.environ['RUNG12_WORK'])
except Exception as e:
    os.environ['RUNG12_WORK'] = '/content/rung12_work'; os.makedirs('/content/rung12_work', exist_ok=True)
    print('[CELL 2] Drive NOT mounted (', type(e).__name__, ') — PDBs in /content (lost on disconnect)')
os.environ['RUNG12_N'] = '32'
from scripts.runlog import new_log, run_logged
import sys
RUNLOG = new_log('rung12_pmhc', repo_dir='.')
run_logged([sys.executable,'-u','scripts/37_pmhc_discriminability.py','prep'], RUNLOG)

In [ ]:
#@title Cell 3 — VALIDATE the scoring logic (synthetic, instant)
import sys
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/37_pmhc_discriminability.py','selftest'], RUNLOG)
print('[CELL 3]', '✓ validated' if rc==0 else '✗ FAILED — stop here')

In [ ]:
#@title Cell 4 — install ESM (robust) + Biopython, and ESMFold extras (best-effort)
import sys
from scripts.runlog import run_logged
run_logged([sys.executable,'-m','pip','install','-q','fair-esm','biopython'], RUNLOG, label='pip fair-esm biopython')
# ESMFold extras (openfold) often fail to build on Colab — best-effort; embeddings work without it.
!pip install -q 'fair-esm[esmfold]' 'dllogger @ git+https://github.com/NVIDIA/dllogger.git' 'openfold @ git+https://github.com/aqlaboratory/openfold.git@4b41059694619831a7db195b7e0988fc4ff3a307' 2>/dev/null || echo '[CELL 4] esmfold extras failed (OK: structural E will fall back to a position prior)'
print('[CELL 4] ✓ (fair-esm + biopython ready; esmfold best-effort)')

In [ ]:
#@title Cell 5 — ESM-2 embeddings -> per-handle Z (the robust measured signal)
import os, json, numpy as np, torch, esm
WORK = os.environ['RUNG12_WORK']
man = json.load(open('runs/rung12_pmhc/rung12_manifest.json'))
model, alphabet = esm.pretrained.esm2_t33_650M_UR50D(); model = model.eval()
dev = 'cuda' if torch.cuda.is_available() else 'cpu'; model = model.to(dev)
bc = alphabet.get_batch_converter()
def embed(seq):
    _,_,toks = bc([('x', seq)]); toks = toks.to(dev)
    with torch.no_grad():
        rep = model(toks, repr_layers=[33])['representations'][33]
    return rep[0, 1:len(seq)+1].mean(0).float().cpu().numpy()
raw = {}
for h in man['handles']:
    if 'groove' not in h: continue
    em, ew = embed(h['pep_mut']), embed(h['pep_wt'])
    cos = float(np.dot(em, ew)/(np.linalg.norm(em)*np.linalg.norm(ew)+1e-9))
    raw[h['id']] = 1.0 - cos
v = np.array(list(raw.values())); lo, hi = float(v.min()), float(v.max())
Z = {k: ((x-lo)/(hi-lo) if hi > lo else 0.0) for k, x in raw.items()}   # min-max normalise across handles
json.dump(Z, open(f'{WORK}/esm_deltas.json', 'w'))
print('[CELL 5] ✓ ESM-2 embedding deltas for', len(Z), 'handles ->', f'{WORK}/esm_deltas.json')

In [ ]:
#@title Cell 6 — ESMFold structures (best-effort, resumable: groove:peptide -> PDB)
import os, glob, json, torch
WORK = os.environ['RUNG12_WORK']
try:
    import esm
    fold = esm.pretrained.esmfold_v1().eval()
    if torch.cuda.is_available(): fold = fold.cuda()
    try: fold.set_chunk_size(128)
    except Exception: pass
    fas = sorted(glob.glob(f'{WORK}/*.fasta')); done = 0
    for i, fa in enumerate(fas):
        pdb = fa[:-6] + '.pdb'
        if os.path.exists(pdb): done += 1; continue
        seq = open(fa).read().splitlines()[1]   # 'groove:peptide' — ':' = ESMFold multimer chainbreak
        try:
            with torch.no_grad(): out = fold.infer_pdb(seq)
            open(pdb, 'w').write(out); done += 1
            if i % 8 == 0: print(f'  folded {i+1}/{len(fas)}')
        except Exception as e:
            print('  fold FAIL', os.path.basename(fa), type(e).__name__, e)
    print(f'[CELL 6] ✓ ESMFold: {done}/{len(fas)} PDBs (resumable)')
except Exception as e:
    print('[CELL 6] ESMFold UNAVAILABLE — skipping structure; analyze will use position-prior E. (', type(e).__name__, e, ')')

In [ ]:
#@title Cell 7 — ANALYZE: measured beta + ranked targets + bridge re-run
import sys, os, json
from scripts.runlog import run_logged
rc = run_logged([sys.executable,'-u','scripts/37_pmhc_discriminability.py','analyze'], RUNLOG)
print('[CELL 7]', '✓ done' if rc==0 else f'✗ exit {rc}')
from IPython.display import Image, display
if os.path.exists('runs/rung12_pmhc/rung12_pmhc.png'): display(Image('runs/rung12_pmhc/rung12_pmhc.png'))
if os.path.exists('runs/rung12_pmhc/rung12_pmhc.json'):
    d = json.load(open('runs/rung12_pmhc/rung12_pmhc.json'))
    print('with-structure:', d['n_with_structure'], '/', d['n_handles'],
          '| per-cell-safe', d['n_per_cell_safe'], '| relay-safe', d['n_relay_safe'],
          '| unlocked', d['n_unlocked_by_relay'])

In [ ]:
#@title Cell 8 — bundle ONE zip + download
import os, glob, zipfile
from scripts.runlog import finalize
finalize(RUNLOG, download=False)
stem = os.path.basename(str(RUNLOG)).replace('rung12_pmhc_','').replace('.log','')
b = f'/content/rung12_run_{stem}.zip'
ps = sorted(set(glob.glob('runs/rung12_pmhc/*.png')+glob.glob('runs/rung12_pmhc/*.json')+[str(RUNLOG)]))
with zipfile.ZipFile(b, 'w', zipfile.ZIP_DEFLATED) as z:
    for x in ps:
        if os.path.exists(x): z.write(x, arcname=os.path.basename(x)); print(' bundled', x)
print('[CELL 8] ->', b)
try:
    from google.colab import files; files.download(b)
except Exception as e: print('(download skipped:', e, ')')